# LeetCode #105 — Construct Binary Tree from Preorder and Inorder Traversal: chia để trị với hash map

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, Divide and Conquer, Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
```

Mục tiêu của notebook:

1. Hiểu vai trò của hai cách duyệt: **preorder** cho biết **gốc**, **inorder** cho biết **cái gì bên trái / bên phải**.
2. Đệ quy chia để trị trên **khoảng chỉ số** (không cắt list).
3. Dùng **hash map** `giá trị → vị trí trong inorder` để từ `O(n²)` xuống `O(n)`.
4. Cách lặp bằng stack (tham khảo) và vì sao cần giá trị **phân biệt**.

# 1. Bài toán là gì?

Cho hai mảng `preorder` và `inorder` là kết quả duyệt **tiền thứ tự** và **trung thứ tự** của **cùng một** cây nhị phân. Dựng lại và trả về cây.

- Preorder: **gốc** → cây trái → cây phải.
- Inorder: cây trái → **gốc** → cây phải.

**Constraints (theo đề):**

- `1 <= preorder.length <= 3000`, `inorder.length == preorder.length`
- `-3000 <= preorder[i], inorder[i] <= 3000`
- Mọi giá trị **phân biệt**.
- Mọi giá trị của `inorder` đều có trong `preorder`; đảm bảo hai mảng là preorder/inorder của cùng một cây.

---

## Ví dụ

```text
preorder = [3, 9, 20, 15, 7]
inorder  = [9, 3, 15, 20, 7]
->            3
             / \
            9  20         [3, 9, 20, null, null, 15, 7]
              /  \
             15   7

preorder = [-1], inorder = [-1]  ->  [-1]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Đệ quy (recursion)
3. Chia để trị (divide and conquer)
4. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.3 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [11]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [12]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.4 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [13]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] =

 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [14]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 263.68 ms | trên set: 0.075 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [15]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [16]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.5 Áp dụng vào bài này

Cần nắm chắc **preorder** (gốc đứng **đầu**) và **inorder** (gốc đứng **giữa**, trái bên trái, phải bên phải). Lấy gốc từ preorder, tìm vị trí của nó trong inorder (tra **hash map** `O(1)`) để biết kích thước cây con trái, rồi **chia để trị** dựng hai cây con bằng **đệ quy**.

# 3. Nền tảng

In [17]:
from typing import List, Optional
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def serialize(root: Optional[TreeNode]) -> list:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def preorder_of(root):
    return [root.val] + preorder_of(root.left) + preorder_of(root.right) if root else []


def inorder_of(root):
    return inorder_of(root.left) + [root.val] + inorder_of(root.right) if root else []

t = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))
print("preorder:", preorder_of(t), "| inorder:", inorder_of(t), "| level-order:", serialize(t))

preorder: [3, 9, 20, 15, 7] | inorder: [9, 3, 15, 20, 7] | level-order: [3, 9, 20, None, None, 15, 7]


# 4. Ý tưởng chính

1. `preorder[0]` là **gốc** (preorder luôn thăm gốc trước).
2. Tìm vị trí `k` của gốc trong `inorder`. Mọi thứ **bên trái** `k` thuộc cây con trái, **bên phải** thuộc cây con phải. Số nút cây trái là `left_size = k − (đầu đoạn inorder)`.
3. Trong preorder, sau gốc là **`left_size` phần tử của cây trái**, rồi tới cây phải.
4. Đệ quy cho hai phía.

```text
preorder = [3 | 9 | 20 15 7]        inorder = [9 | 3 | 15 20 7]
            gốc trái  phải                     trái gốc  phải
left_size = 1
```

# 5. Baseline: cắt list và dùng `index`

Viết thẳng theo ý tưởng: dễ hiểu, nhưng mỗi tầng **cắt list** (`O(n)` sao chép) và `inorder.index(...)` (`O(n)` tìm kiếm) → xấu nhất `O(n²)` (cây lệch).

In [18]:
def build_slicing(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    if not preorder:
        return None
    root_val = preorder[0]
    k = inorder.index(root_val)
    root = TreeNode(root_val)
    root.left = build_slicing(preorder[1:k + 1], inorder[:k])
    root.right = build_slicing(preorder[k + 1:], inorder[k + 1:])
    return root

print(serialize(build_slicing([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

[3, 9, 20, None, None, 15, 7]


Ở đây `left_size = k` vì `inorder` đã được cắt (bắt đầu từ 0).

# 6. Lời giải chính: khoảng chỉ số + hash map

- `pos[v]` = vị trí của `v` trong `inorder` → tìm gốc trong `O(1)`.
- Không cắt list: truyền khoảng `[in_lo, in_hi]` của inorder, và dùng một con trỏ **toàn cục** `pre_idx` chạy dọc preorder.

Vì sao một con trỏ đủ? Preorder là "gốc → toàn bộ cây trái → toàn bộ cây phải". Nếu đệ quy dựng **gốc, rồi trái, rồi phải** (đúng thứ tự preorder), thì các giá trị được tiêu thụ **đúng thứ tự** của `preorder`.

```text
build(in_lo, in_hi):
    if in_lo > in_hi: return None
    root_val = preorder[pre_idx]; pre_idx += 1
    k = pos[root_val]
    root.left  = build(in_lo, k - 1)     # PHẢI dựng trái trước
    root.right = build(k + 1, in_hi)
```

In [19]:
def build_hash(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    pre_idx = 0

    def build(in_lo, in_hi):
        nonlocal pre_idx
        if in_lo > in_hi:
            return None
        root_val = preorder[pre_idx]
        pre_idx += 1
        k = pos[root_val]
        root = TreeNode(root_val)
        root.left = build(in_lo, k - 1)
        root.right = build(k + 1, in_hi)
        return root

    return build(0, len(inorder) - 1)

print(serialize(build_hash([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

[3, 9, 20, None, None, 15, 7]


# 7. Mô phỏng

In [20]:
def trace_build(preorder: List[int], inorder: List[int]) -> list:
    pos = {v: i for i, v in enumerate(inorder)}
    pre_idx = 0
    def build(in_lo, in_hi, depth):
        nonlocal pre_idx
        ind = "  " * depth
        if in_lo > in_hi:
            return None
        root_val = preorder[pre_idx]
        k = pos[root_val]
        print(f"{ind}inorder[{in_lo}..{in_hi}] = {inorder[in_lo:in_hi + 1]} | preorder[{pre_idx}] = {root_val} là gốc"
              f" | trái {inorder[in_lo:k]}, phải {inorder[k + 1:in_hi + 1]}")
        pre_idx += 1
        root = TreeNode(root_val)
        root.left = build(in_lo, k - 1, depth + 1)
        root.right = build(k + 1, in_hi, depth + 1)
        return root
    root = build(0, len(inorder) - 1, 0)
    print("Cây:", serialize(root))
    return serialize(root)

_ = trace_build([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print()
_ = trace_build([1, 2, 4, 5, 3, 6], [4, 2, 5, 1, 6, 3])

inorder[0..4] = [9, 3, 15, 20, 7] | preorder[0] = 3 là gốc | trái [9], phải [15, 20, 7]
  inorder[0..0] = [9] | preorder[1] = 9 là gốc | trái [], phải []
  inorder[2..4] = [15, 20, 7] | preorder[2] = 20 là gốc | trái [15], phải [7]
    inorder[2..2] = [15] | preorder[3] = 15 là gốc | trái [], phải []
    inorder[4..4] = [7] | preorder[4] = 7 là gốc | trái [], phải []
Cây: [3, 9, 20, None, None, 15, 7]

inorder[0..5] = [4, 2, 5, 1, 6, 3] | preorder[0] = 1 là gốc | trái [4, 2, 5], phải [6, 3]
  inorder[0..2] = [4, 2, 5] | preorder[1] = 2 là gốc | trái [4], phải [5]
    inorder[0..0] = [4] | preorder[2] = 4 là gốc | trái [], phải []
    inorder[2..2] = [5] | preorder[3] = 5 là gốc | trái [], phải []
  inorder[4..5] = [6, 3] | preorder[4] = 3 là gốc | trái [6], phải []
    inorder[4..4] = [6] | preorder[5] = 6 là gốc | trái [], phải []
Cây: [1, 2, 3, 4, 5, 6]


# 8. Cách 3 (tham khảo): lặp bằng stack

Duyệt preorder; stack giữ "đường đi" các nút **chưa có con phải**. Con trỏ `j` chạy trên inorder:

- Nếu đỉnh stack **khác** `inorder[j]`: nút mới là **con trái** của đỉnh.
- Ngược lại: pop khi đỉnh bằng `inorder[j]` (và tăng `j`); nút mới là **con phải** của nút pop cuối cùng.

In [21]:
def build_iterative(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    root = TreeNode(preorder[0])
    stack = [root]
    j = 0
    for v in preorder[1:]:
        node = TreeNode(v)
        if stack[-1].val != inorder[j]:
            stack[-1].left = node
        else:
            parent = None
            while stack and stack[-1].val == inorder[j]:
                parent = stack.pop()
                j += 1
            parent.right = node
        stack.append(node)
    return root

print(serialize(build_iterative([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

[3, 9, 20, None, None, 15, 7]


Ý tưởng: inorder cho biết **khi nào** một chuỗi "đi sang trái" kết thúc (khi gặp nút không còn con trái). Cách này `O(n)` và không đệ quy — hữu ích khi cây rất sâu.

# 9. Phiên bản đúng format LeetCode

In [22]:
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        pos = {v: i for i, v in enumerate(inorder)}
        pre_idx = 0

        def build(in_lo: int, in_hi: int) -> Optional[TreeNode]:
            nonlocal pre_idx
            if in_lo > in_hi:
                return None
            root_val = preorder[pre_idx]
            pre_idx += 1
            k = pos[root_val]
            root = TreeNode(root_val)
            root.left = build(in_lo, k - 1)
            root.right = build(k + 1, in_hi)
            return root

        return build(0, len(inorder) - 1)

In [23]:
import sys
sys.setrecursionlimit(10000)

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def random_tree(n):
    vals = random.sample(range(-3000, 3001), n)
    it = iter(vals)
    def make(k):
        if k == 0:
            return None
        left = random.randint(0, k - 1)
        node = TreeNode(next(it))
        node.left = make(left)
        node.right = make(k - 1 - left)
        return node
    return make(n)


sol = Solution()
check("Ví dụ 1", serialize(sol.buildTree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])), [3, 9, 20, None, None, 15, 7])
check("Ví dụ 2", serialize(sol.buildTree([-1], [-1])), [-1])
check("Lệch trái", serialize(sol.buildTree([3, 2, 1], [1, 2, 3])), [3, 2, None, 1])
check("Lệch phải", serialize(sol.buildTree([1, 2, 3], [1, 2, 3])), [1, None, 2, None, 3])

random.seed(105)
for _ in range(3000):
    t = random_tree(random.randint(1, 15))
    pre, ino = preorder_of(t), inorder_of(t)
    expected = serialize(t)
    for f in (sol.buildTree, build_slicing, build_iterative):
        assert serialize(f(pre, ino)) == expected
print("3000 cây ngẫu nhiên: 3 cách dựng lại đúng cây gốc")

chain_pre = list(range(3000)); chain_in = list(range(2999, -1, -1))   # cây lệch trái 3000 nút
r = sol.buildTree(chain_pre, chain_in)
check("Cây lệch 3000 nút", preorder_of(r) == chain_pre and inorder_of(r) == chain_in, True)

Ví dụ 1            PASS  [3, 9, 20, None, None, 15, 7]
Ví dụ 2            PASS  [-1]
Lệch trái          PASS  [3, 2, None, 1]
Lệch phải          PASS  [1, None, 2, None, 3]


3000 cây ngẫu nhiên: 3 cách dựng lại đúng cây gốc
Cây lệch 3000 nút  PASS  True


(Cây lệch 3000 nút có độ sâu đệ quy 3000, vượt giới hạn mặc định 1000 của Python — nên cell trên tăng `sys.setrecursionlimit`. Trên LeetCode điều này thường đã được thiết lập; cách lặp ở mục 8 tránh được hoàn toàn.)

# 10. Vì sao đúng? Độ phức tạp?

**Quy nạp theo kích thước đoạn.** Với đoạn inorder `[in_lo, in_hi]` là inorder của một cây con `T`, và `pre_idx` đang trỏ tới đầu preorder của `T`:

- `preorder[pre_idx]` là gốc của `T`.
- Giá trị phân biệt ⇒ `k = pos[gốc]` xác định duy nhất; `inorder[in_lo..k−1]` là inorder của cây trái, `inorder[k+1..in_hi]` của cây phải.
- Sau khi lấy gốc, `pre_idx` trỏ tới đầu preorder của cây trái; dựng xong cây trái thì `pre_idx` vừa đúng tới đầu preorder của cây phải.

```text
Time:  O(n)   — mỗi nút tạo một lần, tra pos O(1)
Space: O(n)   — hash map; đệ quy O(h)
Baseline cắt list: O(n²) xấu nhất
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Dựng cây **phải trước** trái khi dùng con trỏ `pre_idx`

In [24]:
def wrong_right_first(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    pre_idx = 0
    def build(in_lo, in_hi):
        nonlocal pre_idx
        if in_lo > in_hi:
            return None
        root = TreeNode(preorder[pre_idx]); pre_idx += 1
        k = pos[root.val]
        root.right = build(k + 1, in_hi)          # SAI: preorder tiêu thụ trái trước
        root.left = build(in_lo, k - 1)
        return root
    return build(0, len(inorder) - 1)

try:
    print("Sai :", serialize(wrong_right_first([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))
except Exception as e:
    print("Sai : lỗi", type(e).__name__, e)
print("Đúng:", serialize(sol.buildTree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

Sai : lỗi IndexError list index out of range
Đúng: [3, 9, 20, None, None, 15, 7]


(Thứ tự **phải trước** chỉ đúng khi dựng từ **postorder đọc ngược** — Bài 3.)

## Lỗi 2: Sai biên khi cắt preorder

Cây trái trong preorder là `preorder[1 : 1 + left_size]` (với list đã cắt thì `left_size = k`, nên là `preorder[1 : k + 1]`). Viết `preorder[1 : k]` làm thiếu một phần tử, và phần thừa bị đẩy sang cây phải — nơi nó không có trong `inorder` tương ứng.

In [25]:
def wrong_slice(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    if not preorder:
        return None
    k = inorder.index(preorder[0])
    root = TreeNode(preorder[0])
    root.left = wrong_slice(preorder[1:k], inorder[:k])           # SAI: thiếu một phần tử
    root.right = wrong_slice(preorder[k:], inorder[k + 1:])
    return root

try:
    print("Sai :", serialize(wrong_slice([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))
except ValueError as e:
    print("Sai : lỗi ValueError —", e)
print("Đúng:", serialize(sol.buildTree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))

Sai : lỗi ValueError — 9 is not in list
Đúng: [3, 9, 20, None, None, 15, 7]


## Lỗi 3: Giá trị trùng

Nếu có giá trị trùng, `pos[v]` / `inorder.index(v)` không xác định duy nhất vị trí gốc → có thể có **nhiều** cây cho cùng cặp traversal. Đề đảm bảo giá trị phân biệt.

In [26]:
a = TreeNode(1, TreeNode(1))         # 1 có con trái 1
b = TreeNode(1, None, TreeNode(1))   # 1 có con phải 1
print("Cây a:", preorder_of(a), inorder_of(a), "| Cây b:", preorder_of(b), inorder_of(b))

Cây a: [1, 1] [1, 1] | Cây b: [1, 1] [1, 1]


Hai cây khác nhau có cùng preorder `[1, 1]` và cùng inorder `[1, 1]` — không thể phân biệt.

## Lỗi 4: Dùng `inorder.index` trong mỗi lời gọi

Đúng nhưng `O(n²)` với cây lệch. Dựng `pos` một lần.

# 12. Mẹo nhận diện pattern

**Dựng cây từ hai cách duyệt:** một cách cho biết **gốc** (preorder: đầu; postorder: cuối), inorder cho biết **chia trái/phải**.

| Bài | Gốc ở đâu | Thứ tự dựng |
|---|---|---|
| #105 Preorder + Inorder | đầu preorder | gốc → trái → phải |
| #106 Inorder + Postorder | cuối postorder | gốc → **phải** → trái (đọc postorder ngược) |
| #889 Preorder + Postorder | đầu preorder | không duy nhất nếu có nút một con |
| #1008 BST from Preorder | đầu preorder | inorder = sorted(preorder) |
| #297 Serialize/Deserialize | — | lưu cả `None` thì chỉ cần một cách duyệt |

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Dựng cây từ `preorder = [1, 2, 4, 5, 3, 6, 7]`, `inorder = [4, 2, 5, 1, 6, 3, 7]`.

## Bài 2 — Điền chỗ trống

In [27]:
def exercise_2_build_tree(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    pre_idx = 0
    def build(in_lo, in_hi):
        nonlocal pre_idx
        # TODO: trường hợp rỗng; lấy gốc từ preorder; tìm k; dựng trái rồi phải
        pass
    return build(0, len(inorder) - 1)


# assert serialize(exercise_2_build_tree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])) == [3, 9, 20, None, None, 15, 7]
# assert serialize(exercise_2_build_tree([-1], [-1])) == [-1]

## Bài 3 — Biến thể: Construct from Inorder and Postorder (LeetCode #106)

```python
build_from_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) -> [3, 9, 20, None, None, 15, 7]
```

Gợi ý nhanh: gốc là phần tử **cuối** của postorder; con trỏ chạy **ngược** từ cuối; dựng **phải trước**.

In [28]:
def exercise_3_build_from_post(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# assert serialize(exercise_3_build_from_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])) == [3, 9, 20, None, None, 15, 7]
# assert serialize(exercise_3_build_from_post([-1], [-1])) == [-1]

## Bài 4 — Biến thể: Construct BST from Preorder (LeetCode #1008)

Chỉ cho `preorder` của một **BST** (giá trị phân biệt). Dựng cây trong `O(n)` **không** sắp xếp: dùng cận trên như #98.

```python
bst_from_preorder([8, 5, 1, 7, 10, 12]) -> [8, 5, 10, 1, 7, None, 12]
```

In [29]:
def exercise_4_bst_from_preorder(preorder: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# assert serialize(exercise_4_bst_from_preorder([8, 5, 1, 7, 10, 12])) == [8, 5, 10, 1, 7, None, 12]
# assert serialize(exercise_4_bst_from_preorder([1, 3])) == [1, None, 3]

## Bài 5 — Mini challenge: từ preorder và inorder suy ra postorder (không dựng cây)

Trả về postorder trực tiếp, `O(n)`, không tạo `TreeNode`.

```python
postorder_from([3, 9, 20, 15, 7], [9, 3, 15, 20, 7]) -> [9, 15, 7, 20, 3]
```

In [30]:
def exercise_5_postorder_from(preorder: List[int], inorder: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_5_postorder_from([3, 9, 20, 15, 7], [9, 3, 15, 20, 7]) == [9, 15, 7, 20, 3]
# assert exercise_5_postorder_from([1], [1]) == [1]

# 14. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_build` sau khi tự làm.

**Bài 3**
- Mức 1: `post_idx = len(postorder) - 1`, giảm dần.
- Mức 2: postorder đọc ngược là "gốc → phải → trái".

**Bài 4**
- Mức 1: `build(bound)`: nếu hết preorder hoặc `preorder[i] > bound` thì `None`.
- Mức 2: lấy gốc, `left = build(root.val)`, `right = build(bound)`.

**Bài 5**
- Mức 1: như `build_hash`, nhưng thay vì tạo nút, sau khi xử lý trái và phải thì `out.append(root_val)`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [31]:
_ = trace_build([1, 2, 4, 5, 3, 6, 7], [4, 2, 5, 1, 6, 3, 7])

inorder[0..6] = [4, 2, 5, 1, 6, 3, 7] | preorder[0] = 1 là gốc | trái [4, 2, 5], phải [6, 3, 7]
  inorder[0..2] = [4, 2, 5] | preorder[1] = 2 là gốc | trái [4], phải [5]
    inorder[0..0] = [4] | preorder[2] = 4 là gốc | trái [], phải []
    inorder[2..2] = [5] | preorder[3] = 5 là gốc | trái [], phải []
  inorder[4..6] = [6, 3, 7] | preorder[4] = 3 là gốc | trái [6], phải [7]
    inorder[4..4] = [6] | preorder[5] = 6 là gốc | trái [], phải []
    inorder[6..6] = [7] | preorder[6] = 7 là gốc | trái [], phải []
Cây: [1, 2, 3, 4, 5, 6, 7]


## Đáp án Bài 2

In [32]:
def answer_2_build_tree(preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
    return build_hash(preorder, inorder)

assert serialize(answer_2_build_tree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])) == [3, 9, 20, None, None, 15, 7]
assert serialize(answer_2_build_tree([-1], [-1])) == [-1]
print("Bài 2 PASS (giống build_hash ở mục 6)")

Bài 2 PASS (giống build_hash ở mục 6)


## Đáp án Bài 3

In [33]:
def postorder_of(root):
    return postorder_of(root.left) + postorder_of(root.right) + [root.val] if root else []

def answer_3_build_from_post(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1
    def build(in_lo, in_hi):
        nonlocal post_idx
        if in_lo > in_hi:
            return None
        root = TreeNode(postorder[post_idx]); post_idx -= 1
        k = pos[root.val]
        root.right = build(k + 1, in_hi)
        root.left = build(in_lo, k - 1)
        return root
    return build(0, len(inorder) - 1)

assert serialize(answer_3_build_from_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])) == [3, 9, 20, None, None, 15, 7]
assert serialize(answer_3_build_from_post([-1], [-1])) == [-1]
for _ in range(2000):
    t = random_tree(random.randint(1, 12))
    assert serialize(answer_3_build_from_post(inorder_of(t), postorder_of(t))) == serialize(t)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [34]:
def answer_4_bst_from_preorder(preorder: List[int]) -> Optional[TreeNode]:
    i = 0
    def build(bound):
        nonlocal i
        if i == len(preorder) or preorder[i] > bound:
            return None
        root = TreeNode(preorder[i]); i += 1
        root.left = build(root.val)
        root.right = build(bound)
        return root
    return build(float("inf"))

assert serialize(answer_4_bst_from_preorder([8, 5, 1, 7, 10, 12])) == [8, 5, 10, 1, 7, None, 12]
assert serialize(answer_4_bst_from_preorder([1, 3])) == [1, None, 3]
for _ in range(2000):
    pre = random.sample(range(100), random.randint(1, 12))
    # tạo preorder hợp lệ của một BST: chèn lần lượt rồi lấy preorder
    root = None
    def ins(node, v):
        if not node:
            return TreeNode(v)
        if v < node.val: node.left = ins(node.left, v)
        else: node.right = ins(node.right, v)
        return node
    for v in pre:
        root = ins(root, v)
    p = preorder_of(root)
    assert serialize(answer_4_bst_from_preorder(p)) == serialize(build_hash(p, sorted(p)))
print("Bài 4 PASS (khớp cách preorder + inorder = sorted(preorder))")

Bài 4 PASS (khớp cách preorder + inorder = sorted(preorder))


## Đáp án Bài 5

In [35]:
def answer_5_postorder_from(preorder: List[int], inorder: List[int]) -> List[int]:
    pos = {v: i for i, v in enumerate(inorder)}
    pre_idx = 0
    out = []
    def walk(in_lo, in_hi):
        nonlocal pre_idx
        if in_lo > in_hi:
            return
        root_val = preorder[pre_idx]; pre_idx += 1
        k = pos[root_val]
        walk(in_lo, k - 1)
        walk(k + 1, in_hi)
        out.append(root_val)
    walk(0, len(inorder) - 1)
    return out

assert answer_5_postorder_from([3, 9, 20, 15, 7], [9, 3, 15, 20, 7]) == [9, 15, 7, 20, 3]
assert answer_5_postorder_from([1], [1]) == [1]
for _ in range(2000):
    t = random_tree(random.randint(1, 12))
    assert answer_5_postorder_from(preorder_of(t), inorder_of(t)) == postorder_of(t)
print("Bài 5 PASS")

Bài 5 PASS


# 16. Tổng kết

> Gốc = phần tử tiếp theo của preorder; vị trí của nó trong inorder (tra bằng hash map) chia đoạn thành trái/phải. Đệ quy trên **khoảng chỉ số**, dựng **trái trước phải** để con trỏ preorder chạy đúng. `O(n)` thời gian và bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Dựng `pos` một lần (không gọi `index` mỗi lần)?
- [ ] Dựng cây trái **trước** cây phải (với preorder)?
- [ ] Biên đoạn: `[in_lo, k − 1]` và `[k + 1, in_hi]`?
- [ ] Nhớ rằng cần giá trị phân biệt?